<a href="https://colab.research.google.com/github/umaimafawad711-code/Computer_Vision/blob/main/Week%203/compare_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics -q

In [ ]:
import cv2 as cv
from ultralytics import YOLO
import matplotlib.pyplot as plt

# Settings
FRAME_LIMIT = 50
videos_to_test = {
    "Original": "Video.mp4",
    "Noise": "noise_video.mp4",
    "Blur": "blur_video.mp4",
    "Dark": "dark_video.mp4"
}

models = {
    "YOLO11n": "yolo11n.pt",
    "YOLO11x": "yolo11x.pt"
}

conf_levels = [0.2, 0.4, 0.6, 0.8]
results = {}

# Testing Loop
for model_name, model_path in models.items():
    print(f"\nLoading Model: {model_name}...")
    model = YOLO(model_path)
    results[model_name] = {}

    for video_name, video_path in videos_to_test.items():
        print(f"  Testing {video_name} video...")
        results[model_name][video_name] = {}

        for conf in conf_levels:
            results[model_name][video_name][conf] = 0
            cap = cv.VideoCapture(video_path)

            frames_processed = 0
            while True:
                ret, frame = cap.read()
                if not ret: break

                frame = cv.resize(frame, (640, 360))
                res = model(frame, classes=[0], conf=conf, verbose=False)

                results[model_name][video_name][conf] += len(res[0].boxes)

                frames_processed += 1
                if frames_processed >= FRAME_LIMIT: break

            cap.release()

print("\nPlotting graphs...")

# Graph Plotting
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('YOLO Comparison: Detections vs Confidence (Lower is Harder)', fontsize=16)
positions = [(0,0), (0,1), (1,0), (1,1)]

for idx, video_name in enumerate(videos_to_test.keys()):
    row, col = positions[idx]
    ax = axes[row, col]

    for model_name in models.keys():
        x_values = conf_levels
        y_values = [results[model_name][video_name][c] for c in conf_levels]
        ax.plot(x_values, y_values, marker='o', label=model_name, linewidth=2)

    ax.set_title(f'Video: {video_name}')
    ax.set_xlabel('Confidence')
    ax.set_ylabel('Persons Detected')
    ax.set_xticks(conf_levels)
    ax.legend()
    ax.grid(True, linestyle='--')

plt.tight_layout()
plt.subplots_adjust(top=0.92)

# Graph save karo
plt.savefig('comparison_graphs.png', dpi=150)
print("✅ Graph is ready : comparison_graphs.png")

plt.show()